<a href="https://colab.research.google.com/github/Bing-CRV/-unsloth-/blob/main/nb/Qwen3_(4B)-GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [4]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

### Unsloth

Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.

We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [ ]:
from unsloth import FastModel
import torch

max_seq_length = 2048   # 上下文长度
lora_rank = 32          # LoRA 秩，统一用这个变量

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference=False,
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9,
)

model = FastModel.get_peft_model(
    model,
    r = lora_rank,                      # 直接用统一的变量
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,         # 按 rank 动态设定（推荐）
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


### GRPO chat template
 A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:
# ==============================
# 系统提示（system_prompt）
# ==============================
# 这个 system_prompt 用于引导模型生成群聊风格的回答
system_prompt = """
你是一个聊天助手。
模仿群聊风格：活泼口语化、简短幽默。
阅读聊天上下文后，生成符合风格的回答。
"""

# ==============================
# chat_template
# ==============================
# 这是 Jinja2 风格的模板，用于渲染 prompt
# 支持：
#   - system_prompt 引导模型整体风格
#   - 遍历多轮聊天（user / assistant）
#   - 可选 add_generation_prompt（这里不涉及 reasoning）
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
        "{{ '{system_prompt}' + eos_token }}"
        "{% set loop_messages = messages[1:] %}"
    "{% else %}"
        "{{ '{system_prompt}' + eos_token }}"
        "{% set loop_messages = messages %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
        "{% if message['role'] == 'user' %}"
            "{{ message['content'] }}\n"
        "{% elif message['role'] == 'assistant' %}"
            "{{ message['content'] + eos_token }}\n"
        "{% endif %}"
    "{% endfor %}"
)

# ==============================
# 替换模板变量
# ==============================
chat_template = chat_template.replace("'{system_prompt}'", f"'{system_prompt}'")

# ==============================
# 绑定到 tokenizer
# ==============================
tokenizer.chat_template = chat_template


### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.

We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [ ]:
import pandas as pd
from datasets import Dataset
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

# -----------------------------
# 文件路径
file_path = r"D:\Code\Python\Turrit\Data\train\T4\data\result\test_part.jsonl"

# 读取 JSONL
dataset = pd.read_json(file_path, lines=True)
dataset = dataset[["prompt", "completion"]]

# -----------------------------
# 定义标签
reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

# -----------------------------
# 初始化模型和 tokenizer
model_name = "unsloth/Qwen3-4B-Base"
model, tokenizer = FastModel.from_pretrained(model_name)

# -----------------------------
# 将聊天记录转换成 Messages 列
def format_messages(row):
    final_prompt = reasoning_start + row["completion"] + reasoning_end + solution_start + row["completion"] + solution_end
    return [
        {"role": "system",    "content": "system_prompt"},
        {"role": "user",      "content": row["prompt"]},
        {"role": "assistant", "content": final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_messages, axis=1)

# -----------------------------
# 拼接成文本字段 'text' 并截断
max_seq_length = 2048

dataset["text"] = dataset["Messages"].apply(
    lambda x: tokenizer.apply_chat_template(x)
)

# 根据字符长度截断
dataset = dataset.loc[dataset["text"].str.len() <= max_seq_length // 2].copy()

# 转成 HuggingFace Dataset
train_dataset = Dataset.from_pandas(dataset[["text"]])

# -----------------------------
# 创建 SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        dataset_text_field="text",      # 指定文本列
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,  # 可调整模拟大 batch
        warmup_steps=5,
        num_train_epochs=2,             # 根据数据量调整
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",             # 8bit 优化器节省显存
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",               # 不使用 wandb
    )
)

# -----------------------------
# 查看准备好的数据示例
print(train_dataset[0])


Check to see if it worked:

Let's now pre fine-tune the model so it follows our custom GRPO formatting!

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59 | Num Epochs = 2 | Total steps = 118
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 66,060,288/4,088,528,384 (1.62% trained)


Step,Training Loss
5,0.644900
10,0.639600
15,0.419300
20,0.389600
25,0.422200
30,0.448400
35,0.475100
40,0.419100
45,0.445300
50,0.328000


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=118, training_loss=0.3486394861997184, metrics={'train_runtime': 170.8892, 'train_samples_per_second': 0.691, 'train_steps_per_second': 0.691, 'total_flos': 2374193075607552.0, 'train_loss': 0.3486394861997184})

Let's check if the model has learnt to follow the custom format:

In [ ]:

# 取前两条消息作为输入
text = tokenizer.apply_chat_template(
    dataset["Messages"].iloc[0][:2],  # 第一条样本的前两条消息
    tokenize=False,
    add_generation_prompt=True        # 必须添加 reason_start 以开始生成
)

# -----------------------------
# 导入 streamer
from transformers import TextStreamer
import torch

# 将文本编码为 tensor
inputs = tokenizer(text, return_tensors="pt").to("cuda")

# -----------------------------
# 生成输出
streamer = TextStreamer(tokenizer, skip_prompt=False)

_ = model.generate(
    **inputs,
    temperature=0,
    max_new_tokens=1024,
    streamer=streamer,
)


Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [ ]:
del dataset               # 删除 dataset 变量，释放 Python 对象占用的内存
torch.cuda.empty_cache()  # 清空 PyTorch GPU 缓存
import gc
gc.collect()              # 强制 Python 垃圾回收，释放未引用的对象占用的内存


0

### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:
import pandas as pd
from datasets import Dataset

# -----------------------------
# 本地 JSONL 文件路径
file_path = r"D:\Code\Python\Turrit\Data\train\T4\data\result\test_part.jsonl"

# 读取 JSONL
df = pd.read_json(file_path, lines=True)

# 只保留 prompt 和 completion
df = df[["prompt", "completion"]]

# -----------------------------
# 转成 HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# 查看前几条数据
print(dataset[0])


In GSM8K, ee notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

Let's map the dataset! and see the first row:

In [ ]:
# 假设已经定义了 system_prompt、reasoning_start、solution_start、solution_end
def map_to_grpo(x):
    # 构建 prompt 消息
    prompt_msgs = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ]

    # 构建 answer / completion 消息，加 GRPO 标签
    answer_msg = reasoning_start + x["completion"] + reasoning_end + \
                 solution_start + x["completion"] + solution_end

    return {
        "prompt": prompt_msgs,
        "answer": answer_msg,
    }

# 应用 map
dataset = dataset.map(map_to_grpo)

# 查看第一条数据
print(dataset[0])


We create a regex format to match the reasoning sections and answers:

In [ ]:
import re

# 假设你已经定义
reasoning_start = "<start_working_out>"
reasoning_end   = "<end_working_out>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

# 可选 EOS token 匹配
solution_end_regex = rf"{re.escape(solution_end)}[\s]*" + \
    f"(?:{re.escape(tokenizer.eos_token)})?"

# 编译正则，匹配 reasoning + solution
match_format = re.compile(
    rf"{re.escape(reasoning_start)}"   # 匹配 <start_working_out>
    r"(.*?)"                           # 捕获 reasoning 内容
    rf"{re.escape(reasoning_end)}"     # 匹配 <end_working_out>
    r"\s*"                             # 可选空格/换行
    rf"{re.escape(solution_start)}"    # 匹配 <SOLUTION>
    r"(.*?)"                           # 捕获 solution 内容
    rf"{solution_end_regex}",           # 匹配 </SOLUTION> + 可选 EOS
    flags = re.MULTILINE | re.DOTALL   # 跨行匹配
)

# 测试示例
text = "<start_working_out>我分析了一下<end_working_out><SOLUTION>最终答案是42</SOLUTION>"
match = match_format.search(text)
if match:
    reasoning_text = match.group(1).strip()
    solution_text  = match.group(2).strip()
    print("Reasoning:", reasoning_text)
    print("Solution:", solution_text)


re.compile(r'<end_working_out>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|endoftext\|>)?[\s]{0,}$',
re.MULTILINE|re.DOTALL|re.UNICODE)

We verify it works:

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def match_format_exactly(completions, **kwargs):
    """
    Reward function for GRPO-format outputs.
    +3 points if the output exactly contains the reasoning + solution tags.

    completions: list of model outputs, each is a list of dicts with "content" field
    """
    scores = []
    for completion in completions:
        score = 0
        # 取第一条生成内容
        response = completion[0]["content"]

        # 如果匹配到 reasoning + solution 格式，则奖励 3 分
        if match_format.search(response) is not None:
            score += 3.0

        scores.append(score)
    return scores

# 使用示例
completions = [
    [{"content": "<start_working_out>分析内容<end_working_out><SOLUTION>答案42</SOLUTION>"}],
    [{"content": "没有任何标签的文本"}]
]

print(match_format_exactly(completions))
# 输出: [3.0, 0.0]


If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def match_format_approximately(completions, **kwargs):
    """
    Partial reward function for GRPO-format outputs.
    Reward for following reasoning + solution tags partially.

    completions: list of model outputs, each is a list of dicts with "content" field
    """
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]

        # <start_working_out> 不奖励，因为总会 prepend
        # reasoning_end 出现一次加 0.5，否则 -1
        score += 0.5 if response.count(reasoning_end) == 1 else -1.0
        # <SOLUTION> 出现一次加 0.5，否则 -1
        score += 0.5 if response.count(solution_start) == 1 else -1.0
        # </SOLUTION> 出现一次加 0.5，否则 -1
        score += 0.5 if response.count(solution_end) == 1 else -1.0

        scores.append(score)
    return scores

# 使用示例
completions = [
    [{"content": "<start_working_out>分析内容<end_working_out><SOLUTION>答案42</SOLUTION>"}],
    [{"content": "<start_working_out>只写了分析"}],
    [{"content": "完全没有标签"}]
]

print(match_format_approximately(completions))
# 输出示例: [1.5, 0.5, -3.0]


Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def check_answer(prompts, completions, answers, **kwargs):
    """
    Reward or penalize model outputs based on extracted answers in GRPO format.

    prompts: list of tokenized prompts / chat messages
    completions: list of model outputs, each is a list of dicts with "content"
    answers: list of true answers corresponding to each prompt
    """
    # 获取用户最后一条内容
    question = prompts[0][-1]["content"] if prompts else ""

    # 提取模型生成文本
    responses = [completion[0]["content"] for completion in completions]

    # 尝试使用 match_format 提取 <SOLUTION> 中的答案
    extracted_responses = [
        (guess.group(2).strip() if (guess := match_format.search(r)) is not None else None)
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answers):
        score = 0
        if guess is None:
            # 未匹配到 solution 标签，扣分
            scores.append(-2.0)
            continue

        # 精确匹配答案
        if guess == true_answer:
            score += 5.0
        # 去除空格后匹配
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # 尝试数字相近匹配
            try:
                ratio = float(guess) / float(true_answer)
                if 0.9 <= ratio <= 1.1:
                    score += 2.0
                elif 0.8 <= ratio <= 1.2:
                    score += 1.5
                else:
                    score -= 2.5
            except:
                score -= 4.5  # 非数字或无法转换
        scores.append(score)

    return scores

# 使用示例
completions = [
    [{"content": "<start_working_out>分析<end_working_out><SOLUTION>42</SOLUTION>"}],
    [{"content": "<start_working_out>分析<end_working_out><SOLUTION>43</SOLUTION>"}],
    [{"content": "没有标签"}]
]

answers = ["42", "42", "42"]

print(check_answer([[]], completions, answers))
# 输出示例: [5.0, 1.5, -2.0]


We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_responses(prompts, completions, answer, **kwargs):
    """
    针对聊天记录任务的 reward 函数：
    - 打印问题、真实答案和模型生成内容
    - 奖励模型是否遵循 GRPO 标签格式
    """
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    scores = []
    # 每隔 PRINT_EVERY_STEPS 打印一次
    global PRINTED_TIMES
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"\nQuestion:\n{question}"
            f"\nAnswer:\n{answer[0]}"
            f"\nResponse:\n{responses[0]}"
        )
    PRINTED_TIMES += 1

    for response, true_answer in zip(responses, answer):
        score = 0
        # 检查 GRPO 标签完整性
        score += 0.5 if response.count("<end_working_out>") == 1 else -1.0
        score += 0.5 if response.count("<SOLUTION>") == 1 else -1.0
        score += 0.5 if response.count("</SOLUTION>") == 1 else -1.0
        # 可选：内容匹配真实答案（可用部分匹配、文本相似度等）
        if true_answer.strip() in response:
            score += 1.0
        scores.append(score)
    return scores


Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [ ]:
import numpy as np

# 对聊天记录应用 tokenization，并添加生成提示
tokenized = pretrain_dataset.map(
    lambda x: {
        "tokens": tokenizer.apply_chat_template(
            x["prompt"],
            add_generation_prompt=True,
            tokenize=True
        )
    },
    batched=True,
)

# 打印第一个样本，检查 token 化效果
print(tokenizer.decode(tokenized[0]["tokens"]))

# 计算每条记录的 token 长度
tokenized = tokenized.map(lambda x: {"L": len(x["tokens"])})

# 获取长度的 90% 分位数，作为最大长度
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length =", maximum_length)

# 只保留长度不超过 90% 分位数的样本
pretrain_dataset = pretrain_dataset.select(
    np.where(np.array(tokenized["L"]) <= maximum_length)[0]
)

# 清理内存
del tokenized


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = maximum_length + 1  # +1 以防万一
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1,  # 可以设大一些如4做平滑训练
    num_generations = 4,               # 若显存不足可减小
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 100,
    save_steps = 100,
    report_to = "none",
    output_dir = "outputs",
    # 可选训练与评估配置
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)


Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
from trl import GRPOTrainer
import torch

# 初始化 GRPOTrainer
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,       # 完全匹配格式奖励
        match_format_approximately, # 部分匹配格式奖励
        check_answer,               # 根据正确答案奖励
        check_numbers,              # 根据数值正确性奖励
    ],
    args = training_args,
    train_dataset = dataset,       # 使用你的聊天记录数据集

    # 可选的训练与评估
    # new_dataset = dataset.train_test_split(test_size=0.01)
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)

# 开始训练
trainer.train()


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
from vllm import SamplingParams

def grpo_generate(model, tokenizer, user_input, sampling_params=None):
    """
    自动将用户输入包装成 GRPO 模板并生成回答
    """
    # 使用训练时的 chat template 包装
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_input}],
        tokenize=False,
        add_generation_prompt=True  # 会在末尾加上 <start_working_out>
    )

    # 默认 SamplingParams
    if sampling_params is None:
        sampling_params = SamplingParams(
            temperature=1.0,
            top_k=50,
            max_tokens=1024
        )

    # 调用 fast_generate
    output = model.fast_generate(
        [prompt_text],
        sampling_params=sampling_params,
        lora_request=None
    )[0].outputs[0].text

    return output

# 测试
text = "What is the sqrt of 101?"
output = grpo_generate(model, tokenizer, text)
print(output)


And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework="pt") as f:
    # 遍历文件中的每个权重张量
    for key in f.keys():
        tensor = f.get_tensor(key)  # 取出张量
        # 计算该张量中零元素占比
        n_zeros = (tensor == 0).sum() / tensor.numel()
        # 断言张量不是全零的
        assert(n_zeros.item() != tensor.numel())


Now we load the LoRA and test:

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

# Ollama






In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if True: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

In [ ]:
# import subprocess

# subprocess.Popen(["ollama", "serve"])
# import time

# time.sleep(3)  # Wait for a few seconds for Ollama to load!

In [ ]:
print(tokenizer._ollama_modelfile)

In [ ]:
# !ollama create unsloth_model -f ./model/Modelfile

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")
